# Fine-Tuning Gemma 3 270M with Unsloth + QLoRA

**NYP School of Information Technology — Applied AI Series**

Companion notebook to `production_nlp_peft_lora_qlora.ipynb`. That notebook fine-tuned an
*encoder* (`bert-base-uncased`) for classification. This one fine-tunes a small *decoder*
(instruction-following) language model — `google/gemma-3-270m-it`, via its
[Unsloth](https://github.com/unslothai/unsloth) re-upload — using **QLoRA** end-to-end with
the `unsloth` + `trl` stack, which is the dominant recipe for cheaply adapting small/mid-size
LLMs on a single consumer or free-tier GPU.

**What you will do:**

1. Load `unsloth/gemma-3-270m-it` in 4-bit (QLoRA)
2. Attach LoRA adapters with Unsloth's optimized `get_peft_model`
3. Fine-tune on the `mlabonne/guanaco-llama2-1k` instruction dataset with `trl`'s `SFTTrainer`
4. Compare base vs. fine-tuned generations on held-out prompts
5. Save the LoRA adapter (and optionally push to the HuggingFace Hub)

**Designed to run on the free Colab GPU (T4, ~15GB VRAM)** — a 270M-parameter model in
4-bit is tiny (~500MB), so this trains in well under 15 minutes.
Go to `Runtime → Change runtime type → T4 GPU` before you start.

> Adapted from the walkthrough in
> ["How to Fine-Tune Google Gemma 270M with Unsloth and QLoRA"](https://www.codecademy.com/article/how-to-fine-tune-google-gemma-270m-with-unsloth-and-qlora)
> (Codecademy), reorganized to match this course's notebook conventions.

## 0. Environment Setup

`unsloth` pulls in its own pinned versions of `trl`, `peft`, `accelerate`, and
`bitsandbytes`, so install it fresh rather than mixing with whatever Colab preinstalled.
This cell takes ~1-2 minutes.

In [ ]:
!pip install -q -U unsloth

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU detected — go to Runtime > Change runtime type > T4 GPU"
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU name: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

SEED = 42
torch.manual_seed(SEED)

### (Optional) HuggingFace login

`unsloth/gemma-3-270m-it` is **not gated**, so login isn't required to load it. You only
need this if you plan to push your fine-tuned adapter to the Hub in Section 6.

In [ ]:
PUSH_TO_HUB = False  # flip to True if you want to push the adapter at the end

if PUSH_TO_HUB:
    from huggingface_hub import notebook_login
    notebook_login()

## 1. Load the Base Model in 4-bit (QLoRA)

`FastLanguageModel.from_pretrained(..., load_in_4bit=True)` quantizes the frozen backbone
to 4-bit NF4 on load — this is Unsloth's version of the `BitsAndBytesConfig` step from the
BERT/QLoRA notebook, just wrapped in a single call. `dtype=None` lets Unsloth auto-detect
the best compute dtype (bf16 on T4/A100-class GPUs, fp16 otherwise).

In [ ]:
import os
# Unsloth's "fast downloading" (hf_transfer) can silently truncate model.safetensors on
# Colab, which then surfaces as a confusing "file not found" OSError. Force synchronous,
# verified downloads instead. Must be set before `unsloth` is imported.
os.environ["UNSLOTH_STABLE_DOWNLOADS"] = "1"

from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/gemma-3-270m-it"
MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

## 2. Dataset: Instruction Tuning

[`mlabonne/guanaco-llama2-1k`](https://huggingface.co/datasets/mlabonne/guanaco-llama2-1k)
is a 1,000-example subset of the Guanaco instruction-following dataset, pre-formatted as
single `text` strings with prompt/response turns already templated — convenient for a fast
classroom demo, since there's no manual chat-template assembly needed.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("mlabonne/guanaco-llama2-1k", split="train")
print(dataset)
print(dataset[0]["text"][:500])

In [ ]:
# Sanity-check token lengths against MAX_SEQ_LENGTH before training.
sample_texts = [x["text"] for x in dataset.select(range(min(100, len(dataset))))]
tokenized_lengths = [len(tokenizer.encode(text)) for text in sample_texts]
print(f"Average length: {sum(tokenized_lengths) / len(tokenized_lengths):.0f} tokens")
print(f"Max length: {max(tokenized_lengths)} tokens (limit: {MAX_SEQ_LENGTH})")

## 3. Attach LoRA Adapters

Same idea as the `peft.LoraConfig` in the BERT notebook — freeze the backbone, train only
low-rank adapters — but routed through Unsloth's `get_peft_model`, which patches in its
fused kernels and gradient checkpointing for extra speed/memory savings. Note the wider
`target_modules` list: an LLM's decoder block has separate `q/k/v/o` attention projections
*and* a gated MLP (`gate_proj`/`up_proj`/`down_proj`), all of which benefit from adaptation.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                      # rank — ~50MB of trainable weights at this size
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,              # commonly set equal to r
    lora_dropout=0.1,
    bias="none",
    use_gradient_checkpointing="unsloth",  # Unsloth's memory-efficient checkpointing
    random_state=SEED,
)

model.print_trainable_parameters()

## 4. Training with `SFTTrainer`

`trl`'s `SFTTrainer` wraps the standard HF `Trainer` for supervised fine-tuning on raw text
columns — `dataset_text_field="text"` tells it which column already contains the fully
formatted prompt+response strings. `max_steps=100` keeps this to a short classroom-scale
run (~10-15 min on a T4); for a real fine-tune you'd typically train for full epochs instead.

`report_to="none"` is set explicitly so the cell doesn't pause waiting for a W&B login
prompt — flip it to `"wandb"` if you want the same experiment-tracking workflow as the
BERT/LoRA notebook.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,   # effective batch size = 8
    warmup_steps=10,
    max_steps=100,
    learning_rate=2e-4,              # LoRA-scale LR, same rationale as the BERT notebook
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    output_dir="outputs",
    seed=SEED,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    args=training_args,
)

In [ ]:
train_result = trainer.train()
print(train_result.metrics)

## 5. Evaluation: Base vs. Fine-Tuned

Reload a fresh copy of the base model so the comparison is apples-to-apples, then run the
same prompts through both. `FastLanguageModel.for_inference(...)` switches Unsloth into its
faster inference mode (disables training-only optimizations).

In [ ]:
test_prompts = [
    "Explain the concept of machine learning in simple terms.",
    "What are the benefits of using Python for data science?",
    "How does a neural network learn from data?",
]

def generate(model, tok, prompts, max_new_tokens=128, temperature=0.7):
    responses = []
    for prompt in prompts:
        inputs = tok([prompt], return_tensors="pt").to("cuda")
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens, temperature=temperature
        )
        responses.append(tok.decode(outputs[0], skip_special_tokens=True))
    return responses

In [ ]:
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(base_model)

base_responses = generate(base_model, base_tokenizer, test_prompts)
for prompt, response in zip(test_prompts, base_responses):
    print(f"Prompt: {prompt}\nResponse: {response}\n")

In [ ]:
FastLanguageModel.for_inference(model)

finetuned_responses = generate(model, tokenizer, test_prompts)
for prompt, response in zip(test_prompts, finetuned_responses):
    print(f"Prompt: {prompt}\nResponse: {response}\n")

**Discussion prompts:**

- With only 100 training steps on 1k generic instruction examples, how different are the
  base and fine-tuned responses? What would you expect with a domain-specific dataset
  instead (e.g. banking support replies, like Section 1 of the BERT notebook)?
- `r=16` LoRA adapters here are ~50MB versus a ~500MB 4-bit base model. Compare that
  trainable-parameter ratio to what you saw for `bert-base-uncased` — does the ratio grow
  or shrink as models get bigger, and why does that matter for fine-tuning cost?
- Why does QLoRA's memory advantage (Section 6 of the BERT notebook) matter far more here
  than it did for a 110M-parameter encoder?

## 6. Save the Adapter

Saves the LoRA adapter only (a few tens of MB) — not a merged full model. Anyone loading it
later re-attaches these weights on top of `unsloth/gemma-3-270m-it` with `PeftModel`.

In [ ]:
ADAPTER_DIR = "./outputs/gemma-270m-guanaco-lora"

model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Adapter saved to {ADAPTER_DIR}")

if PUSH_TO_HUB:
    HUB_MODEL_ID = "your-username/gemma-270m-guanaco-lora"  # <-- change this
    model.push_to_hub(HUB_MODEL_ID)
    tokenizer.push_to_hub(HUB_MODEL_ID)
    print(f"Pushed to https://huggingface.co/{HUB_MODEL_ID}")
else:
    print("PUSH_TO_HUB is False — set to True above (and log in) to push to the Hub.")